# AI-chip component-spend nowcast & forecast

`pipelines/nowcast` output: 9-quarter history (4 designers), the **Q1 2026 nowcast**, and
the chained **Q2 2026 forecast**. See the module README for method, sources, and limits.
Run `uv run -m pipelines.nowcast all` first.

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

D = "../data/processed/nowcast"
hist = pl.read_csv(f"{D}/nowcast_history.csv")
q1 = pl.read_csv(f"{D}/nowcast_q1_2026.csv")
q2 = pl.read_csv(f"{D}/forecast_q2_2026.csv")
d2 = pl.read_csv(f"{D}/forecast_q2_2026_estimates_detail.csv")
comps = ["Memory", "Logic", "Packaging", "Auxiliary"]
colors = {"Memory": "#4fa8a0", "Logic": "#e0a44a", "Packaging": "#4a5fd0", "Auxiliary": "#d65a9a"}

def qkey(q):
    n, y = q.split(" ")
    return int(y) * 4 + int(n[1])

hist = hist.with_columns(pl.col("Quarter").map_elements(qkey, return_dtype=pl.Int64).alias("_k")).sort("_k")
complete = hist.filter(pl.col("Quarter") != "Q1 2026")
t1 = q1.filter(pl.col("component") == "TOTAL").row(0, named=True)
t2 = q2.filter(pl.col("component") == "TOTAL").row(0, named=True)

## Total component spend: history + Q1 2026 nowcast + Q2 2026 forecast

In [2]:
ht = complete.with_columns(total=sum(pl.col(c) for c in comps))
fig = go.Figure()
fig.add_scatter(x=ht["Quarter"].to_list(), y=ht["total"].to_list(), mode="lines+markers", name="history (actual)")
for label, t, color in [("Q1 2026 nowcast", t1, "crimson"), ("Q2 2026 forecast", t2, "darkorange")]:
    fig.add_scatter(x=[label[:7]], y=[t["reconciled_base"]], mode="markers", name=label,
                    marker=dict(size=11, color=color),
                    error_y=dict(type="data", symmetric=False, array=[t["high"] - t["reconciled_base"]],
                                 arrayminus=[t["reconciled_base"] - t["low"]]))
fig.update_layout(title="AI-chip component spend (NVIDIA/AMD/Google/Amazon): history + nowcast + forecast",
                  yaxis_title="USD billions", height=500)
fig

## Per-component: Q1 2026 nowcast vs Q2 2026 forecast (with bands)

In [3]:
fig = go.Figure()
for label, src in [("Q1 2026", q1), ("Q2 2026", q2)]:
    pc = src.filter(pl.col("component") != "TOTAL")
    fig.add_bar(name=label, x=pc["component"].to_list(), y=pc["reconciled_base"].to_list(),
                error_y=dict(type="data", symmetric=False,
                             array=[h - b for h, b in zip(pc["high"], pc["reconciled_base"])],
                             arrayminus=[b - l for b, l in zip(pc["reconciled_base"], pc["low"])]))
fig.update_layout(barmode="group", title="Reconciled spend by component: Q1 nowcast vs Q2 forecast",
                  yaxis_title="USD billions", height=450)
fig

## Component spend by category — history + Q1 nowcast + Q2 forecast
Stacked breakdown; the Q1 2026 and Q2 2026 bars use the reconciled-base per component.

In [4]:
hist_long = complete.unpivot(index="Quarter", on=comps, variable_name="Component", value_name="spend")
def reco(src, q):
    return src.filter(pl.col("component") != "TOTAL").select(
        Quarter=pl.lit(q), Component=pl.col("component"), spend=pl.col("reconciled_base"))
bars = pl.concat([hist_long.select(["Quarter", "Component", "spend"]), reco(q1, "Q1 2026"), reco(q2, "Q2 2026")],
                 how="vertical_relaxed")
qorder = complete["Quarter"].to_list() + ["Q1 2026", "Q2 2026"]
fig = px.bar(bars, x="Quarter", y="spend", color="Component",
             category_orders={"Quarter": qorder, "Component": comps}, color_discrete_map=colors,
             labels={"spend": "Total component spend (USD billions)"},
             title="AI-chip component spend by category \u2014 history + Q1 nowcast + Q2 forecast")
fig.add_annotation(x="Q1 2026", y=t1["reconciled_base"], text="nowcast", showarrow=True, arrowhead=2, yshift=8)
fig.add_annotation(x="Q2 2026", y=t2["reconciled_base"], text="forecast", showarrow=True, arrowhead=2, yshift=8)
fig.update_layout(height=550)
fig

## Q2 2026: per-source contribution to each estimate
How each family (trend / supply / demand / price / macro / **analyst**) lands per component.

In [5]:
order = ["trend", "supply", "demand", "price", "macro", "analyst", "reconciled"]
fig = px.bar(d2.to_pandas(), x="estimate", y="value_b", facet_col="component",
             color="estimate", category_orders={"estimate": order, "component": comps},
             labels={"value_b": "USD billions"}, title="Q2 2026 estimate by family, per component (base scenario)")
fig.update_yaxes(matches=None)
fig.update_layout(height=450, showlegend=False)
fig

## Computed confidence by component
Confidence is derived from independent **n_anchors** and family **dispersion** (not asserted).
Logic/Packaging/Auxiliary now carry real anchors (TSMC HPC, CoWoS, Broadcom).

In [6]:
conf = pl.concat([
    q1.filter(pl.col("component") != "TOTAL").select(["component", "confidence", "confidence_score", "n_anchors"]).with_columns(target=pl.lit("Q1 2026")),
    q2.filter(pl.col("component") != "TOTAL").select(["component", "confidence", "confidence_score", "n_anchors"]).with_columns(target=pl.lit("Q2 2026")),
])
fig = px.bar(conf.to_pandas(), x="component", y="confidence_score", color="target", barmode="group",
             text="confidence", category_orders={"component": comps},
             labels={"confidence_score": "confidence score (0-1)"},
             title="Computed confidence by component (anchored): Q1 nowcast vs Q2 forecast")
fig.update_traces(textposition="outside")
fig.update_layout(height=420)
fig